# Uno-style constrained unfolding (Uno) (`unfold_uno`)

This notebook applies **Uno-style Lagrange-Newton constrained unfolding** — the Python port of the R
package **Uno** — to a realistic benchmark: detector readings
synthesised from the **Monte-Carlo calculated spectrum
`t4-14-s.txt_1`** of the
[IAEA Compendium](https://www-nds.iaea.org/benchmarks/), a BNCT-like
beam-shaping-assembly spectrum with a thermal group, an epithermal
$1/E$ region and a fast peak.

We use the built-in GSF response functions (10 Bonner spheres, `0in` –
`18in`, 60 energy bins from 1e-9 to ~631 MeV).  Detector readings are
folded with `Detector.get_effective_readings_for_spectra`, the
spectrum is reconstructed with `unfold_uno`, and the result is
compared against the ground truth — which never enters the unfolding.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bssunfold import Detector, RF_GSF
from bssunfold.utils.comparison import compare_spectra

detector = Detector(RF_GSF)
E = detector.E_MeV
names = detector.detector_names
print(f"Detector grid: {detector.n_energy_bins} bins, "
      f"{E[0]:.1e} - {E[-1]:.1f} MeV")
print("Spheres:", ", ".join(names))
detector.plot_response_functions()


## 1. IAEA Compendium reference spectrum → detector readings

The compendium CSV stores 61-point Monte-Carlo spectra on its
own energy grid; `get_effective_readings_for_spectra` folds
the spectrum with the response functions and resamples it
onto the 60-bin detector grid.

In [ ]:
reference_csv = pd.read_csv(
    '../tests/MonteCarlo_Calculated_spectra_from_IAEA_Comp_for_comparison.csv'
)
readings = detector.get_effective_readings_for_spectra(
    reference_csv[['E_MeV', 't4-14-s.txt_1']]
)
print("Effective readings:")
for nm in names:
    print(f"  {nm:>5s}: {readings[nm]:.4g}")

phi_true = np.interp(
    E, reference_csv['E_MeV'].values, reference_csv['t4-14-s.txt_1'].values
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ax = axes[0]
ax.loglog(E, phi_true, "k-", lw=1.5)
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title="IAEA Compendium spectrum t4-14-s.txt_1 (ground truth)")
ax.grid(True, which="both", ls=":", alpha=0.5)

ax = axes[1]
vals = [readings[nm] for nm in names]
ax.bar(np.arange(len(names)), vals, color="steelblue")
ax.set_yscale("log")
ax.set_xticks(np.arange(len(names)))
ax.set_xticklabels(names, rotation=45)
ax.set(xlabel="sphere", ylabel="reading, a.u.",
       title="Effective Bonner-sphere readings")
ax.grid(True, axis="y", ls=":", alpha=0.5)
fig.tight_layout()
plt.show()


## 2. Uno presets

`unfold_uno` solves the constrained NLP
`min 1/2||W(Ax-b)||^2 + lam/2||D2 x||^2 s.t. x >= 0` with two
Uno presets: `filter_sqp` (exact Hessian + Fletcher-Leyffer filter;
for this convex QP the sub-problem is the answer) and
`ipopt_like` (primal-dual interior point).

In [ ]:
result = detector.unfold_uno(readings, save_result=False)

print(f"method    : {result['method']}")
print(f"objective : {result['objective']:.4g}")
print(f"viol      : {result['constraint_violation']:.3g}")
print(f"dual inf  : {result['dual_infeasibility']:.3g}")
print(f"converged : {result['uno_converged']}")

lines_to_plot = [
    ("Uno filterSQP", result['spectrum'], "C1-"),
]


## 3. The two presets and Hessian modes

Compare the `filterSQP` preset with the IPOPT-like interior point
in both Hessian modes (exact and BFGS).

In [ ]:
variants = {
    "filter_sqp (exact)": dict(preset="filter_sqp"),
    "ipopt_like (exact)": dict(preset="ipopt_like", hessian="exact",
                               max_iterations=120, tolerance=1e-6),
    "ipopt_like (bfgs)":  dict(preset="ipopt_like", hessian="bfgs",
                               max_iterations=120, tolerance=1e-6),
}
results_uno = {}
for label, kw in variants.items():
    res = detector.unfold_uno(readings, save_result=False, **kw)
    results_uno[label] = res
    q = compare_spectra(
        res['spectrum'], phi_true,
        metrics=['pearson_r', 'relative_flux_error',
                 'comprehensive_score'],
    )
    print(f"{label:20s}: iters={res['iterations']:3d}  "
          f"obj={res['objective']:.3e}  "
          f"pearson_r={q['pearson_r']:.3f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(E, phi_true, "k-", lw=2, label="IAEA ground truth")
for label, res in results_uno.items():
    ax.loglog(E, res['spectrum'], lw=1.1, label=label)
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title="Uno-style unfolding: presets and Hessian modes")
ax.set_xlim(E[0], 30)
ax.grid(True, which="both", ls=":", alpha=0.35)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()


## Quality assessment

`compare_spectra` reports the reconstruction metrics against the
independently known IAEA Compendium spectrum (used only for
evaluation).

In [ ]:
quality = compare_spectra(
    result['spectrum'], phi_true,
    metrics=["relative_flux_error", "pearson_r", "comprehensive_score",
             "fluence_difference_percent", "dose_difference_percent"],
    energy=E,
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.loglog(E, phi_true, "k-", lw=2, label="IAEA ground truth")
for label, spec, style in lines_to_plot:
    ax.loglog(E, spec, style, lw=1.2, label=label)
ax.set(xlabel="E, MeV", ylabel=r"$\varphi(E)$, cm$^{-2}$s$^{-1}$bin$^{-1}$",
       title="Uno-style constrained unfolding")
ax.set_xlim(E[0], 30)
ax.grid(True, which="both", ls=":", alpha=0.35)
ax.legend(fontsize=9)
fig.tight_layout()
plt.show()

for k, v in quality.items():
    print(f"{k:>26s}: {v}")
